# Exploratory Data Analysis (EDA) Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Exploratory Data Analysis (EDA)**.  
It provides practical examples, code, and small exercises for the main EDA concepts:

1. Distribution analysis  
2. Outlier detection  
3. Missing value analysis  
4. Boxplot analysis  
5. Histogram analysis  
6. Pairwise variable inspection  
7. Correlation matrix  
8. Cross-tabulation  
9. Trend inspection  

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

## Build a Small Example Dataset

We first create a small dataset with numerical, categorical, missing, and time-like information so that we can practice multiple EDA tasks.

In [ ]:
n = 60

df = pd.DataFrame({
    'feature_1': np.random.normal(loc=50, scale=10, size=n),
    'feature_2': np.random.normal(loc=100, scale=20, size=n),
    'feature_3': np.random.exponential(scale=15, size=n),
    'category': np.random.choice(['A', 'B', 'C'], size=n, p=[0.5, 0.35, 0.15]),
    'time_step': np.arange(1, n + 1)
})

# Create a trend-based target variable
df['target'] = 0.8 * df['feature_1'] + 0.2 * df['feature_2'] + 0.1 * df['time_step'] + np.random.normal(0, 5, size=n)

# Add some outliers
df.loc[[5, 18], 'feature_1'] = [110, 125]
df.loc[[10], 'target'] = 160

# Add missing values
df.loc[[3, 7, 25], 'feature_2'] = np.nan
df.loc[[12, 41], 'feature_3'] = np.nan

df.head()

## 1. Distribution Analysis

Distribution analysis helps us understand the general behavior of each variable, including central tendency, spread, skewness, and whether the variable looks symmetric or skewed.

A quick starting point is to compute summary statistics.

In [ ]:
df.describe(include='all')

We can also check skewness and kurtosis for numerical variables.

In [ ]:
numeric_cols = ['feature_1', 'feature_2', 'feature_3', 'target']

shape_summary = pd.DataFrame({
    'Skewness': df[numeric_cols].skew(),
    'Kurtosis': df[numeric_cols].kurt()
})
shape_summary

**Practice:** identify which variable appears most skewed and think about why.

## 2. Outlier Detection

Outliers are observations that differ strongly from the majority of the data. Two common methods are the **z-score** method and the **IQR rule**.

### 2.1 Z-Score Method

For a variable \(x\):

$$
z_i = \frac{x_i - \bar{x}}{s}
$$

A common rule is that values with \(|z| > 3\) may be considered outliers.

In [ ]:
x = df['feature_1']
z_scores = (x - x.mean()) / x.std(ddof=1)
outliers_z = df.loc[np.abs(z_scores) > 3, ['feature_1']]
outliers_z

### 2.2 IQR Rule

$$
IQR = Q_3 - Q_1
$$

Potential outliers satisfy:

$$
x < Q_1 - 1.5(IQR)
$$
or
$$
x > Q_3 + 1.5(IQR)
$$

In [ ]:
q1 = df['feature_1'].quantile(0.25)
q3 = df['feature_1'].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers_iqr = df.loc[(df['feature_1'] < lower_bound) | (df['feature_1'] > upper_bound), ['feature_1']]
lower_bound, upper_bound, outliers_iqr

**Practice:** repeat the same outlier detection for `target`.

## 3. Missing Value Analysis

Missing value analysis checks how much data is missing and where the missing values are located.

For a variable \(j\), the missing ratio is:

$$
MR_j = \frac{m_j}{n}
$$

where:
- \(m_j\) = number of missing values in variable \(j\)
- \(n\) = total number of observations

In [ ]:
missing_counts = df.isna().sum()
missing_ratio = df.isna().mean()

missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Ratio': missing_ratio
})
missing_summary

A simple visual inspection of missing-value positions can also help.

In [ ]:
plt.figure(figsize=(10, 4))
plt.imshow(df.isna(), aspect='auto')
plt.yticks([])
plt.xticks(range(len(df.columns)), df.columns, rotation=45)
plt.title('Missing Value Map')
plt.show()

## 4. Boxplot Analysis

A boxplot summarizes the median, quartiles, spread, and potential outliers of a variable.

The main quantity is:

$$
IQR = Q_3 - Q_1
$$

Whiskers often extend to:

$$
Q_1 - 1.5(IQR), \quad Q_3 + 1.5(IQR)
$$

In [ ]:
plt.figure(figsize=(8, 4))
plt.boxplot(df['feature_1'].dropna(), vert=False)
plt.title('Boxplot of feature_1')
plt.xlabel('feature_1')
plt.show()

**Practice:** generate a boxplot for `target` and compare the spread with `feature_1`.

## 5. Histogram Analysis

A histogram groups continuous data into bins and counts observations in each interval.

For a bin with width \(w\), the frequency density is:

$$
\text{Density} = \frac{f_i}{n \cdot w}
$$

Histograms help reveal skewness, modality, tails, and rough normality.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df['feature_3'].dropna(), bins=12)
plt.title('Histogram of feature_3')
plt.xlabel('feature_3')
plt.ylabel('Frequency')
plt.show()

**Observation:** `feature_3` was generated from an exponential distribution, so it should look right-skewed.

## 6. Pairwise Variable Inspection

Pairwise inspection helps identify linear or nonlinear relationships between two variables.

A useful numerical measure is covariance:

$$
\operatorname{Cov}(X,Y)=\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})
$$

In [ ]:
pair_df = df[['feature_1', 'feature_2', 'feature_3', 'target']].dropna()

plt.figure(figsize=(6, 5))
plt.scatter(pair_df['feature_1'], pair_df['target'])
plt.title('Scatter Plot: feature_1 vs target')
plt.xlabel('feature_1')
plt.ylabel('target')
plt.show()

np.cov(pair_df['feature_1'], pair_df['target'])[0, 1]

**Practice:** inspect the relationship between `feature_2` and `target` using a scatter plot.

## 7. Correlation Matrix

A correlation matrix summarizes pairwise linear relationships among variables.

The Pearson correlation coefficient is:

$$
r_{xy}=\frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum (x_i-\bar{x})^2 \sum (y_i-\bar{y})^2}}
$$

In [ ]:
corr_matrix = df[['feature_1', 'feature_2', 'feature_3', 'target']].corr()
corr_matrix

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(corr_matrix, interpolation='nearest')
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title('Correlation Matrix')
plt.show()

## 8. Cross-Tabulation

Cross-tabulation summarizes the relationship between two categorical variables.

If \(C_{ij}\) is the count of category combination \((A_i, B_j)\), then:

$$
C_{ij}=\text{count}(A_i,B_j)
$$

We first create another categorical variable from the target.

In [ ]:
df['target_level'] = pd.cut(df['target'], bins=[-np.inf, 60, 90, np.inf], labels=['Low', 'Medium', 'High'])
cross_tab = pd.crosstab(df['category'], df['target_level'])
cross_tab

We can also compute row-wise proportions to better interpret the table.

In [ ]:
cross_tab_row_prop = pd.crosstab(df['category'], df['target_level'], normalize='index')
cross_tab_row_prop

## 9. Trend Inspection

Trend inspection is useful for time-indexed or ordered data.

A simple linear trend model is:

$$
y_t = a + bt + \varepsilon_t
$$

where \(b\) is the slope.

A simple smoothing tool is the moving average:

$$
MA_t = \frac{1}{k}\sum_{i=t-k+1}^{t} x_i
$$

In [ ]:
trend_df = df[['time_step', 'target']].copy()
trend_df['moving_average_5'] = trend_df['target'].rolling(window=5).mean()

plt.figure(figsize=(10, 4))
plt.plot(trend_df['time_step'], trend_df['target'], label='Target')
plt.plot(trend_df['time_step'], trend_df['moving_average_5'], label='5-point Moving Average')
plt.title('Trend Inspection of Target Over Time')
plt.xlabel('Time Step')
plt.ylabel('Target')
plt.legend()
plt.show()

We can estimate the linear trend slope using a simple polynomial fit.

In [ ]:
slope, intercept = np.polyfit(trend_df['time_step'], trend_df['target'], 1)
slope, intercept

## 10. Small EDA Summary Table

This table gathers a few useful EDA outputs in one place.

In [ ]:
eda_summary = pd.DataFrame({
    'Variable': numeric_cols,
    'Missing Count': [df[c].isna().sum() for c in numeric_cols],
    'Mean': [df[c].mean() for c in numeric_cols],
    'Std': [df[c].std() for c in numeric_cols],
    'Skewness': [df[c].skew() for c in numeric_cols],
    'Kurtosis': [df[c].kurt() for c in numeric_cols]
})
eda_summary

## 11. Mini Exercises

Try these on your own:

1. Create a histogram and boxplot for `target`.  
2. Detect outliers in `target` using both z-score and IQR.  
3. Compute a correlation matrix after filling missing values.  
4. Create a cross-tab between a new categorical variable and `category`.  
5. Change the moving-average window from 5 to 10 and compare the trend curve.  
6. Add a new variable with strong nonlinear behavior and inspect it with a scatter plot.

These exercises are useful for AI, machine learning, structural engineering, and scientific data analysis.